# Notebook 18 — EDA Feature Engineering Blend

**Purpose**: Blend `submission_xgb_eda_fe.csv` (XGB with 7 new EDA-driven binary features,  
CV AUC = 0.916671) with the best existing submissions.

## New EDA Features (nb17)
| Feature | Description | Churn Lift |
|---------|-------------|------------|
| `is_fiber_no_support` | Fiber optic + 0 security services | **2.40×** |
| `is_echeck_mtm` | Electronic check + month-to-month | **2.02×** |
| `is_high_charge_mtm` | MonthlyCharges >70 + month-to-month | **1.99×** |
| `is_new_mtm` | Tenure ≤12 + month-to-month | **1.94×** |
| `is_senior_alone` | Senior + no partner + no dependents | **1.85×** |
| `n_security_services` | Count of 4 security services (non-linear) | — |
| `contract_ord` | Contract ordinal (0/1/2) | — |

## Blend Options
| Option | Description | Expected LB |
|--------|-------------|-------------|
| **A** | EDA FE alone (standalone) | ~0.914 |
| **B** | Best blend (0.91433) + EDA FE ×2 | **~0.9145+** |
| **C** | EDA FE heavy (EDA×3 + XGB×2 + LGB×1.5) | ~0.9144 |
| **D** | Rank avg: all 5 strong models | ~0.9143+ |
| **E** | 50/50 prob blend: best_blend vs EDA FE | ~0.9143 |

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata
import os, subprocess

IS_KAGGLE = os.path.exists('/kaggle/input')
SUB_DIR   = '/kaggle/working/' if IS_KAGGLE else '../'
TARGET    = 'Churn'

print(f'Environment : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'SUB_DIR     : {SUB_DIR}')

## 1. Load all source submissions

In [ ]:
SUBS = {
    # ── Existing best models ──────────────────────────────────────────────────
    'xgb_50t':      'submission_xgboost_tuned_multiseed_fe.csv',      # LB: 0.91417
    'xgb_100t_6f':  'submission_xgboost_multiseed_fe_100_6.csv',      # LB: 0.91415
    'lgb_100n':     'submission_lightgbm_100n.csv',                   # LB: (in blend → 0.91433)
    'lgb_fe':       'submission_lgbm_tuned_multiseed_fe.csv',         # LB: (nb09)
    'xgb_sp':       'submission_xgb_scaledpos_ensemble.csv',          # LB: nb10
    # ── New: EDA feature engineering ─────────────────────────────────────────
    'xgb_eda':      'submission_xgb_eda_fe.csv',                      # CV: 0.916671 (nb17)
}

preds = {}
ids   = None

for name, fname in SUBS.items():
    path = os.path.join(SUB_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        if ids is None:
            ids = df[['id']].copy()
        preds[name] = df[TARGET].values
        print(f'  ✓ {name:15s} | mean={preds[name].mean():.4f}  range=[{preds[name].min():.4f},{preds[name].max():.4f}]')
    else:
        print(f'  ✗ {name:15s} | NOT FOUND: {path}')

print(f'\nLoaded {len(preds)}/{len(SUBS)} submissions  |  test rows: {len(ids):,}')

## 2. Correlation matrix

Low correlation between models = high ensemble diversity = bigger gain.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

pred_df = pd.DataFrame(preds)
corr    = pred_df.corr().round(4)
print('Prediction correlation matrix:')
print(corr.to_string())

# Highlight EDA FE vs others
print('\nCorrelation of xgb_eda with each model:')
if 'xgb_eda' in pred_df.columns:
    print(corr['xgb_eda'].drop('xgb_eda').to_string())

## 3. Build all blend options

In [ ]:
def weighted_prob_blend(preds_dict, weights):
    """Weighted average of probability predictions."""
    total_w = sum(weights.values())
    out = np.zeros(len(next(iter(preds_dict.values()))))
    for name, w in weights.items():
        if name in preds_dict:
            out += preds_dict[name] * (w / total_w)
    return out

def rank_avg_blend(preds_dict, weights=None):
    """Rank averaging — robust to scale differences."""
    if weights is None:
        weights = {k: 1.0 for k in preds_dict}
    total_w = sum(weights.values())
    n = len(next(iter(preds_dict.values())))
    out = np.zeros(n)
    for name, p in preds_dict.items():
        w = weights.get(name, 1.0)
        out += rankdata(p) * (w / total_w)
    return out / n   # normalise to [0,1]

blends = {}

# ── Option A: EDA FE standalone ───────────────────────────────────────────────
if 'xgb_eda' in preds:
    blends['A_eda_alone'] = preds['xgb_eda'].copy()

# ── Option B: Best blend weights + EDA FE ×2 ─────────────────────────────────
# Current best: xgb_50t×2 + xgb_100t×2 + lgb_100n×1.5 + xgb_sp×1  → 0.91433
# Adding xgb_eda at weight 2 (equal to other XGB models)
w_B = {'xgb_50t': 2.0, 'xgb_100t_6f': 2.0, 'lgb_100n': 1.5, 'xgb_sp': 1.0, 'xgb_eda': 2.0}
blends['B_best_plus_eda_prob'] = weighted_prob_blend(
    {k: preds[k] for k in w_B if k in preds}, w_B)
blends['B_best_plus_eda_rank'] = rank_avg_blend(
    {k: preds[k] for k in w_B if k in preds}, w_B)

# ── Option C: EDA FE heavy blend ──────────────────────────────────────────────
# Bet big on the new features; EDA×3, backed by proven XGB and LGB
w_C = {'xgb_eda': 3.0, 'xgb_50t': 2.0, 'lgb_100n': 1.5, 'lgb_fe': 1.0}
blends['C_eda_heavy_prob'] = weighted_prob_blend(
    {k: preds[k] for k in w_C if k in preds}, w_C)
blends['C_eda_heavy_rank'] = rank_avg_blend(
    {k: preds[k] for k in w_C if k in preds}, w_C)

# ── Option D: Equal rank avg of all 5 strong models ──────────────────────────
strong = {k: preds[k] for k in ['xgb_50t','xgb_100t_6f','lgb_100n','xgb_sp','xgb_eda']
          if k in preds}
blends['D_rank_avg_5models'] = rank_avg_blend(strong)

# ── Option E: 50/50 prob blend — current_best vs EDA FE ──────────────────────
if 'xgb_eda' in preds and all(k in preds for k in ['xgb_50t','xgb_100t_6f','lgb_100n','xgb_sp']):
    current_best = weighted_prob_blend(
        {k: preds[k] for k in ['xgb_50t','xgb_100t_6f','lgb_100n','xgb_sp']},
        {'xgb_50t': 2.0, 'xgb_100t_6f': 2.0, 'lgb_100n': 1.5, 'xgb_sp': 1.0})
    blends['E_fiftyfifty_prob'] = 0.5 * current_best + 0.5 * preds['xgb_eda']

print(f'Built {len(blends)} blend options:')
for name, arr in blends.items():
    print(f'  {name:30s} | mean={arr.mean():.4f}  range=[{arr.min():.4f},{arr.max():.4f}]')

## 4. Save all blend CSVs

In [ ]:
saved_files = {}

for name, arr in blends.items():
    out = ids.copy()
    out[TARGET] = arr
    fname = f'submission_eda_blend_{name}.csv'
    fpath = os.path.join(SUB_DIR, fname)
    out.to_csv(fpath, index=False)
    saved_files[name] = fpath
    print(f'Saved → {fpath}')

print(f'\n{len(saved_files)} files ready for submission.')

## 5. Submit Options

Run the cell for each option you want to submit.  
**Recommended order**: B_rank → D → B_prob → C_rank → A (standalone)

---
### Option A — EDA FE Standalone
*Baseline to see raw EDA FE score. Expect ~0.914x*

In [ ]:
# ── OPTION A: EDA FE standalone ───────────────────────────────────────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['A_eda_alone'],
    '-m', 'nb17 XGB EDA FE standalone | is_fiber_no_support+is_echeck_mtm+is_new_mtm+is_high_charge_mtm+is_senior_alone | 50 models | CV=0.916671',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

---
### Option B — Best Existing Blend + EDA FE ×2 (RECOMMENDED FIRST SUBMIT)
*Adds EDA FE at weight 2 to the current best blend (LB=0.91433). Best bet for improvement.*

In [ ]:
# ── OPTION B (PROB): best blend + EDA FE — probability average ───────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['B_best_plus_eda_prob'],
    '-m', 'Blend: XGB-50t×2+XGB-100t×2+LGB×1.5+XGB-SP×1+XGB-EDA×2 | prob-avg | EDA FE added',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

In [ ]:
# ── OPTION B (RANK): best blend + EDA FE — rank average ──────────────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['B_best_plus_eda_rank'],
    '-m', 'Blend: XGB-50t×2+XGB-100t×2+LGB×1.5+XGB-SP×1+XGB-EDA×2 | rank-avg | EDA FE added',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

---
### Option C — EDA FE Heavy Blend
*Bets more weight on the new features. Riskier but potentially higher ceiling.*

In [ ]:
# ── OPTION C (PROB): EDA heavy — probability average ─────────────────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['C_eda_heavy_prob'],
    '-m', 'Blend: XGB-EDA×3+XGB-50t×2+LGB-100n×1.5+LGB-FE×1 | prob-avg | EDA-heavy',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

In [ ]:
# ── OPTION C (RANK): EDA heavy — rank average ─────────────────────────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['C_eda_heavy_rank'],
    '-m', 'Blend: XGB-EDA×3+XGB-50t×2+LGB-100n×1.5+LGB-FE×1 | rank-avg | EDA-heavy',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

---
### Option D — Equal Rank Average of 5 Strong Models
*Most diverse blend. Equal weight across XGB-50t, XGB-100t-6f, LGB-100n, XGB-SP, XGB-EDA.*

In [ ]:
# ── OPTION D: equal rank avg of 5 strong models ───────────────────────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['D_rank_avg_5models'],
    '-m', 'Rank-avg 5 models: XGB-50t+XGB-100t-6f+LGB-100n+XGB-SP+XGB-EDA | equal weight',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

---
### Option E — 50/50 Blend: Current Best vs EDA FE
*Conservative hedge. Preserves current best score while incorporating EDA signal.*

In [ ]:
# ── OPTION E: 50/50 prob blend ────────────────────────────────────────────────
r = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'playground-series-s6e3',
    '-f', saved_files['E_fiftyfifty_prob'],
    '-m', '50/50 blend: current-best-blend vs XGB-EDA-FE | conservative hedge',
], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr)

---
## 6. Quick sanity check — prediction distributions

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (name, arr) in zip(axes, blends.items()):
    ax.hist(arr, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted Churn Probability')
    ax.axvline(arr.mean(), color='red', linestyle='--', linewidth=1.5, label=f'mean={arr.mean():.3f}')
    ax.legend(fontsize=8)

# Hide unused
for ax in axes[len(blends):]:
    ax.set_visible(False)

plt.suptitle('Prediction distributions — EDA FE Blend Options', fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = os.path.join(SUB_DIR, 'eda_blend_distributions.png')
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Plot saved → {plot_path}')

---
## Score Tracking

Update this table after each submission:

| Option | File | Public LB | Notes |
|--------|------|-----------|-------|
| Baseline | `submission_blend_custom_rank.csv` | **0.91433** | Current best |
| A | `submission_eda_blend_A_eda_alone.csv` | — | EDA FE standalone |
| B (prob) | `submission_eda_blend_B_best_plus_eda_prob.csv` | — | Best+EDA prob |
| B (rank) | `submission_eda_blend_B_best_plus_eda_rank.csv` | — | Best+EDA rank |
| C (prob) | `submission_eda_blend_C_eda_heavy_prob.csv` | — | EDA heavy prob |
| C (rank) | `submission_eda_blend_C_eda_heavy_rank.csv` | — | EDA heavy rank |
| D | `submission_eda_blend_D_rank_avg_5models.csv` | — | Equal rank avg |
| E | `submission_eda_blend_E_fiftyfifty_prob.csv` | — | 50/50 hedge |

**Recommended submit order** (priority order):
1. **B_rank** — safest bet, adds EDA to current best
2. **D** — most diverse, often rewarded by LB
3. **A** — see standalone score
4. **B_prob** — compare with B_rank
5. **C_rank / E** — if daily submissions allow